# Clase 2 · Notebook 01
# Preparación de datos con Pandas y Scikit-Learn

**Introducción al Deep Learning — Módulo 1, Unidad 2: Preparación de datos y herramientas**

La calidad de un modelo depende, ante todo, de la calidad de sus datos. En este cuaderno aprenderás el
flujo completo de **preparación de datos** sobre un caso real: el dataset de **cáncer de mama de Wisconsin**.

Recorreremos:

1. Cargar los datos en un `DataFrame` de Pandas.
2. Analizarlos: `head`, `info`, `value_counts`, `describe`.
3. Estudiar **correlaciones** entre atributos (Pearson).
4. **Normalizar** los valores (estándar y z-score).
5. **Codificar** atributos categóricos (label, one-hot).
6. Generar los conjuntos de **entrenamiento y test** (muestreo aleatorio vs. estratificado).

> 💡 Ejecuta las celdas en orden con **Shift + Enter**.

## 1. Cargar los datos

En el temario se descarga el CSV original con `wget`. Aquí usamos la **misma base de datos de Wisconsin**
incluida en Scikit-Learn (`load_breast_cancer`), para que el cuaderno funcione sin depender de la red.

La etiqueta (clase) indica si el tumor es **maligno** o **benigno**.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.datasets import load_breast_cancer

data = load_breast_cancer()
df = pd.DataFrame(data.data, columns=data.feature_names)
df["target"] = data.target   # 0 = maligno, 1 = benigno

print("Instancias y atributos:", df.shape)
print("Clases:", dict(zip(range(len(data.target_names)), data.target_names)))

## 2. Analizar la estructura de los datos

### 2.1 `head` — primeras filas
Pandas trabaja con dos estructuras: **Series** (una columna) y **DataFrame** (tabla de Series).

In [ ]:
df.head()

### 2.2 `info` — tipos y valores nulos
`info()` muestra el tipo de cada atributo y cuántos valores no nulos tiene (clave para detectar **falta de información**).

In [ ]:
df.info()

### 2.3 `value_counts` — distribución de la clase
Muy útil para atributos discretos y para comprobar si las clases están **balanceadas**.

In [ ]:
conteo = df["target"].value_counts()
print(conteo)
print("\nProporción por clase:")
print((df["target"].value_counts(normalize=True) * 100).round(1).astype(str) + " %")

### 2.4 `describe` — estadísticos de los atributos numéricos
Devuelve count, media, desviación típica, mínimo y máximo. Comparando `min`/`max` con la media y la `std`
podemos intuir la presencia de **valores atípicos (outliers)**.

In [ ]:
df.describe()

## 3. Análisis de correlaciones (Pearson)

El **coeficiente de correlación de Pearson** mide la dependencia lineal entre dos variables continuas.
Va de **-1 a 1**: cerca de 1 → fuerte relación directa; cerca de -1 → inversa; cerca de 0 → sin relación lineal.
Sirve para detectar atributos **redundantes**.

In [ ]:
# Matriz de correlación de un subconjunto de atributos (para que sea legible)
cols = ["mean radius", "mean texture", "mean perimeter", "mean area", "mean smoothness"]
corr = df[cols].corr()
corr.round(2)

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(corr, cmap="coolwarm", vmin=-1, vmax=1)
ax.set_xticks(range(len(cols))); ax.set_xticklabels(cols, rotation=45, ha="right")
ax.set_yticks(range(len(cols))); ax.set_yticklabels(cols)
for i in range(len(cols)):
    for j in range(len(cols)):
        ax.text(j, i, f"{corr.iloc[i, j]:.2f}", ha="center", va="center", fontsize=9)
fig.colorbar(im, label="Pearson")
plt.title("Matriz de correlación")
plt.tight_layout(); plt.show()

Observa que `mean radius`, `mean perimeter` y `mean area` están **muy correlacionados** entre sí
(describen lo mismo: el tamaño del tumor). Mantener los tres aporta poca información nueva.

## 4. Normalización de valores

Los atributos tienen escalas muy distintas (`mean area` vale cientos; `mean smoothness`, menos de 1).
Eso confunde a muchos algoritmos. **Normalizar** los lleva a una escala común.

- **Estándar / Min-Max** → escala 0–1.
- **Z-score** → media 0 y desviación 1 (la más usada en redes de neuronas profundas).

In [ ]:
from sklearn.preprocessing import MinMaxScaler, StandardScaler

X = df[cols].values

minmax = MinMaxScaler().fit_transform(X)     # escala 0-1
zscore = StandardScaler().fit_transform(X)   # media 0, std 1

print("Originales  -> min %.2f  max %.2f" % (X.min(), X.max()))
print("Min-Max     -> min %.2f  max %.2f" % (minmax.min(), minmax.max()))
print("Z-score     -> media %.2f  std %.2f" % (zscore.mean(), zscore.std()))

## 5. Codificar atributos categóricos

Los algoritmos trabajan con números, así que las categorías de texto deben transformarse.
El dataset de Wisconsin es totalmente numérico, así que creamos un atributo categórico de ejemplo.

In [ ]:
from sklearn.preprocessing import LabelEncoder, OneHotEncoder

transporte = np.array(["aéreo", "marítimo", "terrestre", "aéreo", "terrestre"])

# (1) Codificación por etiquetas: un entero por categoría
le = LabelEncoder()
etiquetas = le.fit_transform(transporte)
print("Etiquetas:", etiquetas, "| clases:", list(le.classes_))
print("  Ojo: introduce un orden artificial (aéreo<marítimo<terrestre).\n")

# (2) One-hot: un vector binario, sin orden
oh = OneHotEncoder(sparse_output=False)
onehot = oh.fit_transform(transporte.reshape(-1, 1))
print("One-hot:\n", onehot.astype(int))
print("  Columnas:", list(oh.categories_[0]))

## 6. Generar los conjuntos de entrenamiento y test

Dividimos los datos con `train_test_split`. El **algoritmo de muestreo** importa, sobre todo con
clases desbalanceadas:

- **Aleatorio**: reparte al azar (puede sesgar conjuntos pequeños).
- **Estratificado** (`stratify`): mantiene la proporción de clases en train y test.

In [ ]:
from sklearn.model_selection import train_test_split

X = df[data.feature_names].values
y = df["target"].values

# Muestreo aleatorio
Xtr_a, Xte_a, ytr_a, yte_a = train_test_split(X, y, test_size=0.2, random_state=42)
# Muestreo estratificado
Xtr_s, Xte_s, ytr_s, yte_s = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

def proporcion(y_):
    u, c = np.unique(y_, return_counts=True)
    return {int(k): round(v / len(y_), 3) for k, v in zip(u, c)}

print("Proporción original :", proporcion(y))
print("Test (aleatorio)    :", proporcion(yte_a))
print("Test (estratificado):", proporcion(yte_s))
print("\nTamaños -> train:", Xtr_s.shape[0], "| test:", Xte_s.shape[0])

Fíjate en que el muestreo **estratificado** reproduce mejor la proporción original de clases en el test.

## Resumen

Has preparado un conjunto de datos real de principio a fin:

- **Análisis** con Pandas (`head`, `info`, `value_counts`, `describe`).
- **Correlaciones** para detectar atributos redundantes.
- **Normalización** (Min-Max y Z-score) para igualar escalas.
- **Codificación** de categorías (label y one-hot).
- **Generación de conjuntos** con muestreo aleatorio y estratificado.

➡️ En el **Notebook 02** conoceremos **TensorFlow**, la librería con la que construiremos redes de neuronas.